In [0]:
%run ./opal_functions

In [0]:
cursor = ""
page_size = 5000
ret = get_opal_requests(cursor=cursor, page_size=page_size)
requests_accum = ret.json()['requests']
if 'cursor' in ret.json():
    cursor = ret.json()['cursor']
    while 'cursor' in ret.json():
        print(len(requests_accum))
        ret = get_opal_requests(cursor=cursor, page_size=page_size)
        requests_accum += ret.json()['requests']
        if 'cursor' in ret.json():
            cursor = ret.json()['cursor']

In [0]:
import json
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, ArrayType

# Create a SparkSession
spark = SparkSession.builder.appName("StructTypeExample").getOrCreate()

req_items = StructType([
    StructField("group_id", StringType(), True),
    StructField("name", StringType(), True) 
])

schema = StructType([
    StructField("created_at", StringType(), True),
    StructField("duration_minutes", IntegerType(), True),
    StructField("id", StringType(), True),
    StructField("reason", StringType(), True),
    StructField("requested_items_list", ArrayType(req_items), True),
    StructField("requester_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("target_user_id", StringType(), True),
    StructField("updated_at", StringType(), True)
])

df = spark.createDataFrame(requests_accum, schema)

# Display the DataFrame
#df.show(1, False)
data_expanded=df.select("created_at", "duration_minutes", "id", "reason", "requested_items_list.group_id", "requested_items_list.name", "requester_id", "status", "target_user_id", "updated_at")

In [0]:
df.write.saveAsTable("workspace.opal_dev.opal_demo_requests", mode="overwrite") 
data_expanded.write.saveAsTable("workspace.opal_dev.opal_demo_requests_formatted", mode="overwrite") 

In [0]:
df.show(1, False)

+---------------------------+----------------+------------------------------------+----------+-------------------------+------------------------------------+--------+------------------------------------+---------------------------+
|created_at                 |duration_minutes|id                                  |reason    |requested_items_list     |requester_id                        |status  |target_user_id                      |updated_at                 |
+---------------------------+----------------+------------------------------------+----------+-------------------------+------------------------------------+--------+------------------------------------+---------------------------+
|2025-12-18T22:48:12.797611Z|2880            |669ba7d7-940e-48b4-8b39-1b1f72fdba65|pwease owo|[{NULL, Amalia Bookmark}]|f683c128-da52-4792-aa58-527ce7cdbe29|CANCELED|f683c128-da52-4792-aa58-527ce7cdbe29|2025-12-18T22:48:17.090174Z|
+---------------------------+----------------+--------------------------

In [0]:
df.head(10)

[Row(created_at='2025-12-18T22:48:12.797611Z', duration_minutes=2880, id='669ba7d7-940e-48b4-8b39-1b1f72fdba65', reason='pwease owo', requested_items_list=[Row(group_id=None, name='Amalia Bookmark')], requester_id='f683c128-da52-4792-aa58-527ce7cdbe29', status='CANCELED', target_user_id='f683c128-da52-4792-aa58-527ce7cdbe29', updated_at='2025-12-18T22:48:17.090174Z'),
 Row(created_at='2025-12-17T23:23:13.447751Z', duration_minutes=15, id='9283606a-abcb-4ba5-8cfc-d68b6629eea8', reason='access to query the api for development planning 2', requested_items_list=[Row(group_id=None, name='adrians workspace')], requester_id='9d8efbed-fd36-4d47-a312-37dcf8605345', status='APPROVED', target_user_id='9d8efbed-fd36-4d47-a312-37dcf8605345', updated_at='2025-12-17T23:23:14.273116Z'),
 Row(created_at='2025-12-17T22:41:51.649294Z', duration_minutes=15, id='284c6996-ce44-4d2c-9b7f-7ce8007996e7', reason='access to query the api for development planning', requested_items_list=[Row(group_id=None, name='a